Docs:<br>
- [ReportLab Docs](https://docs.reportlab.com/reportlab/userguide/ch1_intro/)
- [StreamLit Gallery - voor ophalen van data](https://streamlit.io/gallery)

<br>

Basisdingen:
[How to iterate over all or certain columns of a df](https://www.geeksforgeeks.org/python/loop-or-iterate-over-all-or-certain-columns-of-a-dataframe-in-python-pandas/) 
<br>
Om naar te kijken: <br>
-  [Google resultaten concepts](https://www.google.com/search?q=app+pc+for+brainstorming+with+drawing+tablet&num=10&sca_esv=9151e0e90600ee3c&sxsrf=ANbL-n7LjWhKl-hEHRYIRnvv6TYRhGIzTA%3A1774340841409&ei=6UrCaenXGMqLi-gPo-_UgAM&biw=1712&bih=1326&ved=0ahUKEwip8L_cjriTAxXKxQIHHaM3FTAQ4dUDCBE&uact=5&oq=app+pc+for+brainstorming+with+drawing+tablet&gs_lp=Egxnd3Mtd2l6LXNlcnAiLGFwcCBwYyBmb3IgYnJhaW5zdG9ybWluZyB3aXRoIGRyYXdpbmcgdGFibGV0MgUQIRigATIFECEYoAEyBRAhGKABSMY-UABY1j1wBngBkAEAmAFwoAGwHaoBBDQ5LjG4AQPIAQD4AQGYAjigArUfwgILEAAYgAQYkQIYigXCAgoQABiABBhDGIoFwgIQEC4YgAQY0QMYQxjHARiKBcICBRAAGIAEwgILEC4YgAQY0QMYxwHCAgUQLhiABMICBhAAGBYYHsICBxAAGIAEGA3CAgYQABgNGB7CAgsQABiABBiGAxiKBcICBRAAGO8FwgIIEAAYgAQYogTCAgcQIRigARgKwgIFECEYnwXCAgQQIRgVmAMAkgcENTQuMqAH3JsCsgcENDguMrgHkh_CBwkwLjI5LjI2LjHIB6EBgAgA&sclient=gws-wiz-serp)
<br>

Aantekeningen:
<br>

Hoe zorg ik dat ik meerdere inputs df's kan verwerken in één uiteindelijke score?


In [9]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from fpdf import FPDF
import tempfile
from pathlib import Path
import os
import glob

In [52]:
#Tijdelijke oplossing totdat ik andere manier heb gevonden om input te krijgen
folder_path = r'c:\Users\hans_\Documents\GitHub\Stakeholder_analysis\Testdingen'
pattern = os.path.join(folder_path, '*.csv')
csv_files = glob.glob(pattern)


print(csv_files)

['c:\\Users\\hans_\\Documents\\GitHub\\Stakeholder_analysis\\Testdingen\\input_stakeholders.csv', 'c:\\Users\\hans_\\Documents\\GitHub\\Stakeholder_analysis\\Testdingen\\input_stakeholders_2.csv']


In [16]:
#Create function to synthesize assesment input
def synthesize_assessment_input(file_path):
    pattern = os.path.join(file_path, '*.csv')
    csv_files = glob.glob(pattern)

    all_data = []

    if not csv_files:
        return FileNotFoundError(f"No CSV files found in the folder: {folder_path}") 
    #Doorloop alle csv bestanden en voeg ze samen in een dataframe
    for file in csv_files:
        temp_df = pd.read_csv(file, sep=';')
        all_data.append(temp_df)

    #Combineer alle dataframes in één dataframe
    combined_df = pd.concat(all_data, ignore_index=True)

    #Logica om de gecombineerde dataframe te verwerken en te synthetiseren
    aggregated_logic = {
        'formeel': 'mean',
        'informatie': 'mean',
        'informeel': 'mean',
        'legitimiteit': 'mean',
        'betrokkenheid': 'mean',
        'waarom': lambda x: ' | '.join(set(x))
    }

    synthesis = combined_df.groupby('stakeholder').agg(aggregated_logic).reset_index()
    #Mogelijk later toevoegen om af te ronden
    return synthesis

In [17]:
df = synthesize_assessment_input(r'c:\Users\hans_\Documents\GitHub\Stakeholder_analysis\Testdingen')

In [ ]:
#Check for Apple/WINDOWS path issues
print('cwd=', os.getcwd())
print(os.path.exists(r'c:\Users\hans_\Documents\GitHub\Stakeholder_analysis\Testdingen\.ipynb_checkpoints\input_stakeholders-checkpoint.csv'))


cwd= C:\Users\hans_
True


In [53]:
def load_data(file_path):
    try:
        return pd.read_csv(file_path, sep=';')
    except FileNotFoundError:
        return pd.DataFrame({"StakeHolder": ['Project']}) #temp 



In [21]:
#Structureren van data:
def data_structure(df):
    columns = df.columns.tolist()
    for col in columns:
        try:
            all_counts = df[col].value_counts()
            return all_counts
        except ValueError:
            print('wrong values')

In [22]:
def get_strategy(power, interest): #nodig: power en interest score op basis van input)
    if power >= 4 and interest >= 4: return "Manage closely"
    if power >= 4 and interest < 4: return "keep satisfied"
    if power < 4 and interest >= 4: return "keep informed"
    return "Monitor only"


In [23]:
def create_power_interest_columns(dataFrame):
    power_columns = ['formeel', 'informatie', 'informeel','legitimiteit']
    dataFrame['power_scores'] = dataFrame[power_columns].mean(axis=1)
    dataFrame['interest'] = dataFrame['betrokkenheid']

In [24]:
def create_matrix_plot(df):
    fig, ax = plt.subplots(figsize=(6,4))
    ax.scatter(df['power_scores'], df['interest'], c='blue')

    #Kwadranten indelen
    plt.axhline(3, color='black', linewidth=1)
    plt.axvline(3, color='black', linewidth=1)
    plt.xlim(1,5)
    plt.ylim(1,5)

    plt.xlabel('power (1-5)')
    plt.ylabel('interest (1-5)')
    plt.title('Stakeholder map')

    for i, txt in enumerate(df['stakeholder']):
        ax.annotate(txt, (df['power_scores'].iat[i], df['interest'].iat[i])) #.iat werkt als iloc, maar dan voor specifieke cellen, niet hele rijen of kolommen

    plt.tight_layout()
    plt.show()
    plot_path = tempfile.NamedTemporaryFile(delete=False, suffix=".png").name
    plt.savefig(plot_path)
    print(f"File location: {plot_path}")
    return plot_path

In [35]:
df = load_data('Hansgielen92/Stakeholder_analysis/Testdingen/input_stakeholders.csv')
df = pd.read_csv(r'c:\Users\hans_\Documents\GitHub\Stakeholder_analysis\Testdingen\.ipynb_checkpoints\input_stakeholders-checkpoint.csv', sep=';')

In [18]:
print(df)

     stakeholder  formeel  informatie  informeel  legitimiteit  betrokkenheid  \
0            FNV      3.5         4.5        4.0           3.5            3.5   
1  Stakeholder_d      5.0         4.0        5.0           5.0            3.0   
2            VNG      2.0         2.5        3.5           4.0            4.0   
3        VNO-NCW      4.0         1.5        3.0           4.0            3.0   

                             waarom  
0  Heeft veel officiële bevoegdheid  
1                           reden 4  
2                 Reden 4 | Reden 3  
3  Heeft geen officiële bevoegdheid  


In [19]:
columns = df.columns.tolist()
print(columns)

['stakeholder', 'formeel', 'informatie', 'informeel', 'legitimiteit', 'betrokkenheid', 'waarom']


In [38]:
print(help(df.iterrows))


Help on method iterrows in module pandas.core.frame:

iterrows() -> 'Iterable[tuple[Hashable, Series]]' method of pandas.core.frame.DataFrame instance
    Iterate over DataFrame rows as (index, Series) pairs.

    Yields
    ------
    index : label or tuple of label
        The index of the row. A tuple for a `MultiIndex`.
    data : Series
        The data of the row as a Series.

    See Also
    --------
    DataFrame.itertuples : Iterate over DataFrame rows as namedtuples of the values.
    DataFrame.items : Iterate over (column name, Series) pairs.

    Notes
    -----
    1. Because ``iterrows`` returns a Series for each row,
       it does **not** preserve dtypes across the rows (dtypes are
       preserved across columns for DataFrames).

       To preserve dtypes while iterating over the rows, it is better
       to use :meth:`itertuples` which returns namedtuples of the values
       and which is generally faster than ``iterrows``.

    2. You should **never modify** somethi

In [25]:
create_power_interest_columns(df)
df['strategy'] = df.apply(lambda row: get_strategy(row['power_scores'], row['interest']), axis=1)


for i, row in df.iterrows():
    print(f"Stakeholder: {row['stakeholder']}, power: {row['power_scores']}, interest: {row['interest']}, Strategy: {row['strategy']}")


Stakeholder: FNV, power: 3.875, interest: 3.5, Strategy: Monitor only
Stakeholder: Stakeholder_d, power: 4.75, interest: 3.0, Strategy: keep satisfied
Stakeholder: VNG, power: 3.0, interest: 4.0, Strategy: keep informed
Stakeholder: VNO-NCW, power: 3.125, interest: 3.0, Strategy: Monitor only


In [26]:
#Plot stakeholders op kaart
create_matrix_plot(df)


File location: C:\Users\hans_\AppData\Local\Temp\tmpyj90mj_i.png


C:\Users\hans_\AppData\Local\Temp\ipykernel_12180\1052847246.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


'C:\\Users\\hans_\\AppData\\Local\\Temp\\tmpyj90mj_i.png'